## working with agents on top of carteirinha extracted database

### this also should incorporate the base workflow style:
input: blob, id -> image, id -> llm ->  output: convenio, plano, nome da pessoa e número da carteirinha


In [39]:
import os
import base64
import sys
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Tuple
from PIL import Image   # noqa: F401
from pydantic import BaseModel as PydanticBaseModel
from pydantic import Field
from pydantic_settings import BaseSettings
import oracledb
import io
import cv2
import fitz
import numpy as np
import boto3
from botocore.config import Config
import json
import re
import time
from tqdm import tqdm
import mariadb
import pandas as pd
from collections import defaultdict
from dataclasses import dataclass, field
#strands agetinc workflow
from strands.models import BedrockModel
from strands import Agent

#dont limit the visualization of all the colluns of a pandas dataframe
pd.set_option("display.max_columns", None)

# Add project root to Python path so we can import from app module
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.append(project_root)
# Now we can import from app (after adding to sys.path)
from app.utils.logger import get_logger
# Load environment variables from the project root directory
env_path = os.path.join(project_root, '.env')
load_dotenv(env_path)

logger = get_logger(name=__name__)

## credentials config

In [40]:
@dataclass
class AppConstants:
    BEDROCK_DEFAULT_MODEL_ID: str = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
    DEFAULT_PROMPTS_DIR: str = "prompts/"
    S3_BUCKET_NAME: str = "agente-ai-carteirinha"
    S3_RESULTS_PREFIX: str = "resultados"
    S3_DEBUG_PREFIX: str = "debug"
    STREAMING: bool = False
    CACHE_PROMPT = "default"
    RETRIES: Dict[str, int] = field(default_factory=lambda: {"max_attempts": 3, "mode": "standard"})
    CONNECTION_TIMEOUT: int = 5
    READ_TIMEOUT: int = 60
    TEMPERATURE: float = 0.0
    TOP_P: float = 0.1

In [41]:
class Settings(BaseSettings):
    """Carrega e valida as configurações a partir de variáveis de ambiente."""

    ORACLE_USER: str
    ORACLE_PASSWORD: str
    ORACLE_DSN: str
    ORACLE_INSTANT_CLIENT_PATH: Optional[str] = Field(
        None, alias="oracle_instant_client_path"
    )
    AWS_ACCESS_KEY_ID: str
    AWS_SECRET_ACCESS_KEY: str
    AWS_BEDROCK_REGION: str
    BEDROCK_MODEL_ID: str = AppConstants.BEDROCK_DEFAULT_MODEL_ID
    AWS_SERVICE_NAME: str
    MARIADB_USER: str
    MARIADB_PASSWORD: str
    MARIADB_HOST: str
    MARIADB_PORT: int = 3306
    MARIADB_DATABASE: str
    API_BASE_URL: Optional[str] = Field(None, alias="api_base_url")
    API_USERNAME: Optional[str] = Field(None, alias="username")
    API_PASSWORD: Optional[str] = Field(None, alias="password")

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

In [42]:
def create_boto3_session(
    settings: Settings, config: Optional[Config] = None
) -> boto3.Session:
    try:
        logger.info(
            f"Criando sessão boto3 para a região: {settings.AWS_BEDROCK_REGION}..."
        )
        session = boto3.Session(
            region_name=settings.AWS_BEDROCK_REGION,
            aws_access_key_id=settings.AWS_ACCESS_KEY_ID,
            aws_secret_access_key=settings.AWS_SECRET_ACCESS_KEY,
        )
        logger.info("Sessão boto3 criada com sucesso.")
        return session
    except Exception as e:
        logger.critical(f"Não foi possível criar a sessão boto3: {e}")
        raise

## geting the service ready to use
### model configuration

In [43]:

app_constants = AppConstants()
settings = Settings()

# Create a custom boto3 session

session = create_boto3_session(settings)

# Create a Bedrock model with the custom session
bedrock_model = BedrockModel(
    model_id=settings.BEDROCK_MODEL_ID,
    boto_session=session,
    streaming=False,
    temperature=app_constants.TEMPERATURE,
    top_p=app_constants.TOP_P,
    boto_client_config=Config(
        retries=app_constants.RETRIES,
        connect_timeout=app_constants.CONNECTION_TIMEOUT,
        read_timeout=app_constants.READ_TIMEOUT
    )
)

{"timestamp": "2025-08-11T17:28:19", "level": "INFO", "name": "__main__", "message": "Criando sessão boto3 para a região: us-east-1...", "filename": "244079052.py", "lineno": 5}
{"timestamp": "2025-08-11T17:28:19", "level": "INFO", "name": "__main__", "message": "Sessão boto3 criada com sucesso.", "filename": "244079052.py", "lineno": 13}


In [44]:
from strands import Agent

agent = Agent(model=bedrock_model)

prompt = "Tell me about Amazon Bedrock."
response = agent(prompt=prompt)


# Amazon Bedrock

Amazon Bedrock is a fully managed service that provides access to a range of foundation models (FMs) through a unified API. It allows developers to build generative AI applications without having to manage the underlying infrastructure.

Key features include:

- **Multiple model options**: Access to models from leading AI companies including Amazon's own Titan models, Anthropic Claude, Meta Llama 2, AI21 Labs, Cohere, and Stability AI
- **Customization capabilities**: Fine-tune models with your own data
- **Security and privacy**: Your data and customizations remain private and aren't used to train the underlying models
- **Serverless experience**: No infrastructure management required
- **Integration with AWS services**: Works with other AWS tools like SageMaker and Lambda

Bedrock enables various use cases including content generation, summarization, chatbots, search enhancement, and code generation while providing enterprise-grade security and scalability.

In [84]:
# system prompt will be passed as a txt file
class AgentCarteirinha():
    """ This agent will use a multimodal LLM.
    The user will input a list of images on base64, and the agent will process them accordingly, based
    in its agent definition by the system prompt and the passed images.
    The output should be a well structured json"""
    def __init__(self, model: BedrockModel, system_prompt: str, data_model: PydanticBaseModel):
        self.model = model
        self.agent = Agent(model=self.model, system_prompt=system_prompt)
        self.data_model = data_model

    def extract_data_from_images_bytes(self, list_images_bytes: List[bytes]) -> Dict:
        """ This function will receive a list of images in bytes png format,
        and will return a structured json with the extracted data.
        We append max of 20 images and our agent definition as prompt to send
        on a single request."""

        images_text_list = []
        for idx, img_bytes in enumerate(list_images_bytes[:20], start=1):
            images_text_list.append({
                "document": {
                    "format": "png",
                    "name": f"image_{idx}",
                            "source": {
                                "bytes": img_bytes
                            }
                        }
                    })

        # agent response
        return self.agent.structured_output(output_model=self.data_model, prompt=images_text_list)



In [46]:
# image utils
MAX_IMAGES_PER_BLOB = 20  # Maximum number of images per blob
def converter_blob_para_imagens(blob: bytes, extensao: str) -> List[Image.Image]:
    imagens = []
    ext = extensao.lower().strip(".") if extensao else ""
    try:
        if ext == "pdf":
            with fitz.open(stream=blob, filetype="pdf") as pdf_doc:
                logger.info(f"Processando PDF com {len(pdf_doc)} página(s)...")
                for pagina in pdf_doc: # limit to 20 pages
                    if len(imagens) >= MAX_IMAGES_PER_BLOB:
                        logger.warning(f"⚠️ BLOB has reached the maximum limit of {MAX_IMAGES_PER_BLOB} images.")
                        break
                    pix = pagina.get_pixmap(matrix=fitz.Matrix(3.0, 3.0), alpha=False)
                    imagens.append(Image.open(io.BytesIO(pix.tobytes("png"))))
        elif ext in ["jpg", "jpeg", "png", "bmp"]:
            imagens.append(Image.open(io.BytesIO(blob)))
        else:
            logger.warning(f"Formato de arquivo não suportado: '{ext}'.")
    except Exception as e:
        logger.error(f"Erro ao converter BLOB para imagem (ext: .{ext}): {e}")
    return imagens


def aplicar_clahe(imagem: Image.Image) -> Image.Image:
    try:
        imagem_cv = cv2.cvtColor(np.array(imagem), cv2.COLOR_RGB2BGR)
        imagem_cinza = cv2.cvtColor(imagem_cv, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return Image.fromarray(clahe.apply(imagem_cinza))
    except Exception:
        return imagem

def imagens_para_bytes(lista_imagens: List[Image.Image], formato: str = "PNG") -> List[bytes]:
    imagens_bytes = []
    for img in lista_imagens:
        buffer = io.BytesIO()
        img.save(buffer, format=formato)
        imagens_bytes.append(buffer.getvalue())
    return imagens_bytes

## load the dataset and explore for prompt ideas

In [ ]:
import sqlalchemy
data_path = "gold_carteirinha_database.sqlite"
engine = sqlalchemy.create_engine(f"sqlite:///{data_path}")
df_merged_final = pd.read_sql("SELECT * FROM carteirinha", engine)

df_merged_final.info()

In [66]:
# get a dataframe that groups by NOME_CONVENIO
df_grouped = df_merged_final[["NOME_CONVENIO", "NR_CARTEIRA"]].groupby("NOME_CONVENIO").agg(list).reset_index()


# geting a new df  with only the first ocurrency of NR_CARTEIRA FOR EACH NOME_CONVENIO
df_first_occurrence = df_grouped.explode("NR_CARTEIRA").drop_duplicates(subset=["NOME_CONVENIO"], keep="first").reset_index(drop=True)

# NOW lets ADD the len of NR_CARTEIRA col
df_first_occurrence["NR_CARTEIRA_LEN"] = df_first_occurrence["NR_CARTEIRA"].apply(lambda x: len(x) if isinstance(x, str) else 0)

df_first_occurrence = df_first_occurrence.sort_values(by="NR_CARTEIRA_LEN", ascending=False).reset_index(drop=True)

print(df_first_occurrence)

                    NOME_CONVENIO        NR_CARTEIRA  NR_CARTEIRA_LEN
0           SUL AMERICA DIRETO BH  88888474451670021               17
1             STELLANTIS SAUDE MG  00010001091710017               17
2                     SUL AMERICA  88888483405330026               17
3                           CASSI   1101703901120038               16
4         CAIXA ECONOMICA FEDERAL   0101407360000254               16
5                    BLUE COMPANY   0000000620569300               16
6         POSTAL SAUDE - CORREIOS   0184140747000288               16
7                            IPSM   6500116540743112               16
8                  UNIMED SEGUROS   9941868288890028               16
9                        BRADESCO    954560112208012               15
10             BRADESCO OPERADORA    954560109681023               15
11             PLAN ASSISTE - MPF     10511003380000               14
12                      CARE PLUS       090900038401               12
13              PETR

### load a blob to our agent and see the response

In [90]:
# data model
class CarteirinhaExtraida(PydanticBaseModel):
    """Define a estrutura dos dados extraídos da carteirinha."""
    cd_aviso_cirurgia: Optional[str] = Field(None, description="ID do aviso de cirurgia")
    confidence_its_carteirinha: Optional[float] = Field(None, description="Confiança na extração da carteirinha.")
    real_carteirinha_num: Optional[str] = Field(None, description="Número real da carteirinha.")
    convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")
    plano: Optional[str] = Field(None, description="Nome do plano de saúde.")
    nome_pessoa: Optional[str] = Field(None, description="Nome completo do titular ou beneficiário.")
    numero_carteirinha: Optional[str] = Field(None, description="O número de identificação da carteirinha.")

In [55]:
example_blob = df_merged_final.iloc[0]["LO_DOCUMENTO_ANEXO_CIRURGICO"]

images = converter_blob_para_imagens(example_blob, "pdf")
images_with_clahe = [aplicar_clahe(img) for img in images]
images_bytes_list = imagens_para_bytes(images_with_clahe)

{"timestamp": "2025-08-11T17:52:28", "level": "INFO", "name": "__main__", "message": "Processando PDF com 1 página(s)...", "filename": "2847820741.py", "lineno": 9}


In [87]:
# load the query from the docs/carteirinha_agent_prompt_v3.txt
with open("docs/carteirinha_agent_prompt_v3.txt", "r", encoding="utf-8") as file:
    carteirinha_agent_prompt = file.read()


In [89]:
carteirinha_agent_prompt.strip()

'"""\n        You are an expert in OCR and structured data extraction. Your job is to analyze the provided images and extract relevant information about health insurance cards.\n        They are in portuguese so the next instructions are too.\n        Você receberá uma ou mais imagens. Analise todas e verifique se alguma delas contém uma carteirinha de convênio de saúde.\n        Uma carteirinha normalmente possui estes campos:\n        - convenio: O nome da operadora do plano de saúde.\n        - plano: O tipo ou nome do plano (ex: "Plano Prata", "Enfermaria").\n        - nome_pessoa: O nome completo do beneficiário.\n        - numero_carteirinha: O número de identificação ou matrícula da carteirinha. Não deve haver caracter especial, como  es\n        onde, de forma mais tecnica:\n            convenio: Optional[str] = Field(None, description="Nome do convênio de saúde.")\n            plano: Optional[str] = Field(None, description="Nome do plano de saúde.")\n            nome_pessoa: O

In [91]:
# instantiate our agent

carteirinha_agent = AgentCarteirinha(model=bedrock_model, system_prompt=carteirinha_agent_prompt, data_model=CarteirinhaExtraida)

In [92]:
# send the image bytes to agent and get a response
response = carteirinha_agent.extract_data_from_images_bytes(images_bytes_list)

ValueError: Circular reference detected and not supported